In [1]:
from pathlib import Path
from datetime import datetime
import numpy as np
import gymnasium as gym
from matplotlib import pyplot as plt

In [2]:
env = gym.make("LunarLander-v3")
env_wrapper = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=100)


n_episode = 5

for episode in range(n_episode):
    obs, info = env_wrapper.reset()
    done = False
    while not done:
        obs, reward, terminated, truncated, infos = env_wrapper.step(env_wrapper.action_space.sample())
        
        if terminated or truncated:
            done = True
            print(f"Episode Reward: {infos['episode']['r']}")
            print(f"Episode Length: {infos['episode']['l']}")
            print(f"Episode Time: {infos['episode']['t']}")
            print(f"Most recent rewards: {env_wrapper.return_queue}")
            print(f"Most recent episode lengths: {env_wrapper.length_queue}")
            print(f"Most recent episode times: {env_wrapper.time_queue}")

env_wrapper.close()

Episode Reward: -111.68104809534361
Episode Length: 80
Episode Time: 0.004002
Most recent rewards: deque([np.float64(-111.68104809534361)], maxlen=100)
Most recent episode lengths: deque([80], maxlen=100)
Most recent episode times: deque([0.004002], maxlen=100)
Episode Reward: -424.4883573096076
Episode Length: 99
Episode Time: 0.005134
Most recent rewards: deque([np.float64(-111.68104809534361), np.float64(-424.4883573096076)], maxlen=100)
Most recent episode lengths: deque([80, 99], maxlen=100)
Most recent episode times: deque([0.004002, 0.005134], maxlen=100)
Episode Reward: -160.26092552100604
Episode Length: 79
Episode Time: 0.002918
Most recent rewards: deque([np.float64(-111.68104809534361), np.float64(-424.4883573096076), np.float64(-160.26092552100604)], maxlen=100)
Most recent episode lengths: deque([80, 99, 79], maxlen=100)
Most recent episode times: deque([0.004002, 0.005134, 0.002918], maxlen=100)
Episode Reward: -123.10192155212005
Episode Length: 60
Episode Time: 0.00132

In [3]:
envs = gym.vector.SyncVectorEnv(
    [
        lambda: gym.make(
            "LunarLander-v3",
            gravity=np.clip(
                np.random.normal(loc=-10.0, scale=1.0),
                a_min=-11.99,
                a_max=-0.01
            ),
            enable_wind=np.random.choice([True, False]),
            wind_power=np.clip(
                np.random.normal(loc=15.0, scale=1.0),
                a_min=0.01,
                a_max=19.9,
            ),
            turbulence_power=np.clip(
                np.random.normal(loc=1.5, scale=0.5),
                a_min=0.01,
                a_max=1.99,
            )
        )
        for i in range(2)
    ]
)

envs_wrapper = gym.wrappers.vector.RecordEpisodeStatistics(envs)

n_episodes=1
for episode in range(n_episodes):
    obs, infos = envs_wrapper.reset()
    done = [False for _ in range(envs_wrapper.num_envs)]
    while False in done:
        obs, rewards, terminated, truncated, infos = envs_wrapper.step(envs_wrapper.action_space.sample())
        curr_done = [ter or tru for ter, tru in zip(terminated, truncated)]
        done = [cur or do for cur, do in zip(curr_done, done)]
        # print(f"rewards: {rewards}, termindated: {terminated}, truncated: {truncated}, infos: {infos}")
    print(f"most recent rewards: {envs_wrapper.return_queue}")
    print(f"most recent lengths: {envs_wrapper.length_queue}")
    print(f"most times: {envs_wrapper.time_queue}")

envs_wrapper.close()

most recent rewards: deque([np.float64(-54.04385770830986), np.float64(-186.71545751399168)], maxlen=100)
most recent lengths: deque([np.int64(61), np.int64(116)], maxlen=100)
most times: deque([np.float64(0.007928), np.float64(0.014031)], maxlen=100)


In [7]:
import torch

tensor1 = torch.tensor([[1.5, 2.5], [3.5, 4.5]])
print(f"tensor1 shape: {tensor1.shape}, tensor1[0] shape: {tensor1[0].shape}")
print(f"tensor1.detach().cpu().numpy(): {tensor1.detach().cpu().numpy()}, type: {type(tensor1.detach().cpu().numpy())}")

tensor2 = torch.tensor([1, 2, 3, 4, 5, 6, 10, 1])
print(f"torch.max(tensor2): {torch.max(tensor2, dim=-1)}, tensor2.shape: {tensor2.shape}, toorch.max(tensor2, dim=-1).values.shape: {torch.max(tensor2, dim=-1).values.shape}")

tensor1 shape: torch.Size([2, 2]), tensor1[0] shape: torch.Size([2])
tensor1.detach().cpu().numpy(): [[1.5 2.5]
 [3.5 4.5]], type: <class 'numpy.ndarray'>
torch.max(tensor2): torch.return_types.max(
values=tensor(10),
indices=tensor(6)), tensor2.shape: torch.Size([8]), toorch.max(tensor2, dim=-1).values.shape: torch.Size([])


In [14]:
from torch.distributions import Categorical

logits = torch.tensor([
    [
        1, 2, 3,
    ],
    [
        2, 3, 100,
    ]
])
actions = torch.tensor([1, 2])
dist = Categorical(logits=logits)
actions_log_prob = dist.log_prob(actions)
print(f"dist.probs: {dist.probs}")
print(f"dist.sample(): {dist.sample()}")
print(f"dist.log_prob: {dist.log_prob}")
print(f"acitons_log_prob: {actions_log_prob}")

dist.probs: tensor([[9.0031e-02, 2.4473e-01, 6.6524e-01],
        [2.7465e-43, 7.4689e-43, 1.0000e+00]])
dist.sample(): tensor([2, 2])
dist.log_prob: <bound method Categorical.log_prob of Categorical(probs: torch.Size([2, 3]), logits: torch.Size([2, 3]))>
acitons_log_prob: tensor([-1.4076,  0.0000])


In [6]:
tensor3 = torch.tensor([
    [1, 2, 3],
    [2, 3, 4],
    [3, 4, 5]
])
index = torch.tensor([
    [1],
    [2],
    [2]
])
print(f"tensor3.gather(): {tensor3.gather(dim=1, index=index)}")

tensor3.gather(): tensor([[2],
        [4],
        [5]])


In [13]:
actions = np.array([1, 2, 3])
random_actions = np.array([4, 4, 4])
mask = np.array([0, 1, 1], dtype=bool)
actions[mask] = random_actions[mask]
print(actions)

[1 4 4]
